# 04 · Supervisor and brief — close the loop, write for the RM

**What this notebook does:** adds the last two nodes and wires the whole graph for the first time:

```
START → research → signal → policy → supervisor ─┬→ brief → END
             ↑                                    │
             └────────── retry (max 2) ───────────┘
```

- **`supervisor`** reads the evidence gaps and decides: go back to research, or on to the brief. Plain code, no model.
  A routing decision the bank has to defend should be something you can read in ten lines.
- **`brief`** writes what the relationship manager (RM) reads. This is the one place where writing prose is the
  job, so this is where the model does most of its work. It still works on a short leash: code builds the facts it may
  use, checks every sentence cites one, and writes the decision and the clause table itself.

**Where the rules come from:** the EVD policy clauses that notebook 03 *routed* elsewhere land here. EVD-05
(retry limit) is the supervisor's rule; EVD-02, 03, 04, 06 and 07 are the brief's.

## 1 · Setup — the nodes now come from a package

Notebooks 01–03 each built one node inside the notebook. Wiring the full graph needs all of them in one place,
and the supervisor's retry edge means research must be callable from here, not just loaded from a saved
state. So the three nodes moved into `genai/copilot/`, the way `policy_store.py` came out of `06_rag`:

| module | from | changed on the way |
|---|---|---|
| `copilot/research.py` | 01 | the four calls run concurrently; a **failed** call is recorded in `research_errors` instead of crashing the graph |
| `copilot/signals.py` | 02 | none (split into `deterministic_signals` + the model call, so it can be checked without one) |
| `copilot/policy.py` | 03 | a missing record turns its clauses into evidence gaps instead of a `KeyError` |
| `copilot/refs.py` | 02, 03 | the one copy of `charge_refs`; 03's copy is gone |
| `copilot/llm.py` | — | the shared model client and settings |

The second change to research is what makes a retry meaningful. A call that *fails* (timeout, server error)
might succeed next time. A record that comes back empty with *no* error (a 404) is an answer; asking again
returns the same empty record.

In [1]:
import json, operator, sys
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
GENAI = ROOT / "genai"
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(GENAI))

from copilot import llm
from copilot.refs import charge_refs, filing_ref
from copilot.research import load_tools, make_research
from copilot.signals import deterministic_signals, signal
from copilot.policy import policy, CLAUSE

TOOL = await load_tools()
AS_OF = "2026-09-23"                      # fixed, so every run below is reproducible
print("tools:", list(TOOL))

tools: ['get_company_profile', 'get_filing_history', 'get_charges', 'get_officers']


**Did anything change in the move?** Check the package against what the notebooks produced: notebook 02's saved
state for signals, and notebook 03's printed result for 36EL. No model call is needed for either.

In [2]:
STATE_DIR = GENAI / "production" / "state"
for n in ("10812571", "00445790"):
    saved = json.loads((STATE_DIR / f"after_signal_{n}.json").read_text())
    same = deterministic_signals(saved) == {k: v for k, v in saved["signals"].items() if k != "collateral"}
    print(f"{n}  signals identical to notebook 02: {same}")

r = await policy(json.loads((STATE_DIR / "after_signal_10812571.json").read_text()))
print(f"10812571  policy: {r['outcome']}, {len(r['applicable'])} applicable, gaps {r['evidence_gap']}"
      f"   (notebook 03: DECLINE by CON-02, 9 applicable, no gaps)")

10812571  signals identical to notebook 02: True
00445790  signals identical to notebook 02: True
10812571  policy: {'decision': 'DECLINE', 'clauses': ['CON-02']}, 9 applicable, gaps []   (notebook 03: DECLINE by CON-02, 9 applicable, no gaps)


## 2 · The full state — and the one slot that appends

`langgraph-shared-state-framing.md` sorts state fields into four kinds. The schema below follows it:

| kind | slots | who writes |
|---|---|---|
| invariant input | `company_number`, `as_of` | the caller, once |
| evidence | `profile` … `officers`, `research_errors`, `signals`, the five policy slots | research, signal, policy |
| control flow | `research_attempts`, `route`, `supervisor_log` | supervisor |
| terminal output | `brief`, `citations`, `brief_dropped` | brief |

**Reducers.** The framing note suggested `operator.add` for evidence slots "so a Research retry appends, not
clobbers". That's the right worry but the wrong fix here. Each research pass returns the *complete* record,
not a delta, so appending would list every charge twice after one retry and SEC-06 would find "tranches" that
are just duplicates. Evidence is **overwritten** on each pass: the newest complete fetch wins.

The one slot that should append is the **history of what the supervisor decided**, `supervisor_log`.
Each pass adds a line, and overwriting would keep only the last one.

In [3]:
class CopilotState(TypedDict, total=False):
    # invariant input
    company_number: str
    as_of: str
    # evidence — overwritten on every pass (each research pass is a complete re-fetch)
    profile: dict
    filings: dict
    charges: dict
    officers: dict
    research_errors: list[str]
    signals: dict
    applicable: list[dict]
    outcome: dict
    qualifies: bool | None
    evidence_gap: list[str]
    policy_trace: dict
    # control flow — supervisor
    research_attempts: int
    route: str
    supervisor_log: Annotated[list[str], operator.add]     # appends: the history of routing decisions
    # terminal output — brief
    brief: str
    citations: list[str]
    brief_dropped: list[dict]

## 3 · The supervisor — retry only when a retry can help

EVD-05 allows at most two returns to research. The supervisor's job is to spend those only when a retry can change
something. The rules, checked in this order:

| situation | route | why |
|---|---|---|
| no evidence gap | brief | nothing to fetch |
| decision is DECLINE | brief | EVD-07: DECLINE is already the most restrictive; nothing missing could soften it |
| a fetch failed, fewer than 2 retries used | **research** | a timeout or server error may not repeat |
| a fetch failed, 2 retries used | brief | EVD-05's limit |
| gaps, but no fetch failed | brief | the tools would return the same record: Tesco's truncated filing window, a charge whose detail is only an image |

The last row is the one the plan in 03 flagged. Without it, Tesco would use both retries re-fetching the
same 60 filings and end up exactly where it started.

**How EVD-05's heading is applied.** The clause requires *"Thin evidence — RM verification required"* on the
third pass. Row 5 can send a brief out on the *first* pass with gaps still open, and it would be odd for that
brief to look *more* certain than one sent after two retries. So the brief (section 7) uses the heading whenever it
goes out with a gap that leaves the decision open (`qualifies is None`), however many passes that took. This
reads the clause's intent as "a brief with open gaps must say so"; it's a judgement call worth confirming with
whoever owns the policy.

The supervisor is a node that writes `route`, plus a *conditional edge* that reads it. Keeping the decision in
state, not only in the edge function, is what puts it in `supervisor_log` and makes it auditable.

In [4]:
MAX_RETRIES = 2                                                     # EVD-05


def supervisor(state: CopilotState) -> dict:
    gaps, errors = state["evidence_gap"], state.get("research_errors", [])
    tries = state.get("research_attempts", 0)

    if not gaps:
        route, why = "brief", "no evidence gap"
    elif state["qualifies"] is False:
        route, why = "brief", f"{state['outcome']['decision']} — nothing missing could make it less restrictive"
    elif errors and tries < MAX_RETRIES:
        route, why = "research", f"fetch failed ({'; '.join(errors)}) — retry {tries + 1} of {MAX_RETRIES}"
    elif errors:
        route, why = "brief", f"fetch still failing after {MAX_RETRIES} retries (EVD-05)"
    else:
        route, why = "brief", f"{len(gaps)} gap(s), none from a failed fetch — a retry would return the same record"

    return {"route": route,
            "research_attempts": tries + (route == "research"),
            "supervisor_log": [f"pass {tries + 1}: {state['outcome']['decision']:21s} → {route:8s} {why}"]}

## 4 · The brief — who writes what

A brief mixes things a model is good at (turning facts into readable sentences) with things it must never get
wrong (the decision, which clauses apply). So the brief is built in two parts:

| part | written by | why |
|---|---|---|
| title, *Thin evidence* heading, gaps | **code** | EVD-05's heading is exact wording; the gaps are the policy node's list |
| decision line + the clauses that set it | **code** | EVD-06: the outcome must name its clause. It's copied from `outcome`, never paraphrased |
| table of every applicable clause + evidence | **code** | EVD-07: every triggered clause must be cited |
| summary (4–7 statements) | **model** | the part that needs writing |

And the clauses notebook 03 routed here, each enforced by construction rather than by asking nicely:

| clause | how it's enforced |
|---|---|
| EVD-02 every statement sourced | each statement must list fact IDs; code **deletes** any statement with no source or an unknown one, as the clause says, and keeps it in `brief_dropped` for audit |
| EVD-03 no external knowledge | the model sees only the fact sheet; the prompt forbids outside knowledge. (Code can't fully check this, see section 13) |
| EVD-04 collateral only from particulars | a charge's collateral is put on the fact sheet **only if 02's quote check passed**, so an unverified label can't be cited |
| EVD-06, EVD-07 | the code-written parts above |

## 5 · The fact sheet — everything the model may say, keyed by a citable ID

The keys are the same references every node already uses: `profile`, a charge reference like `108125710002`, a
filing reference like `AA filed 2025-12-17`, a clause ID like `CON-02`. So a citation in the brief points straight
at a record, which is what EVD-02 asks for ("traceable to a specific filing or charge record").

What's left out matters as much as what's in. Raw particulars text is omitted; only a *verified* collateral label
with its quote goes in (EVD-04). Filings are cut to accounts and insolvency ones.

In [5]:
def fact_sheet(state: CopilotState) -> dict[str, dict]:
    s, facts = state["signals"], {}
    if p := state.get("profile"):
        facts["profile"] = {k: p[k] for k in ("company_name", "company_number", "company_status", "type",
                                              "date_of_creation", "sic_codes", "accounts_type",
                                              "last_accounts_made_up_to", "next_accounts_due", "accounts_overdue")}
    if s.get("officers"):
        facts["officers"] = s["officers"]
    if ch := state.get("charges"):
        for ref, c in zip(charge_refs(ch["items"]), ch["items"]):
            fact = {k: c[k] for k in ("created_on", "status", "persons_entitled", "lender_group",
                                      "contains_fixed_charge", "contains_floating_charge", "contains_negative_pledge")}
            col = s["collateral"].get(ref)
            if col and col["verified"] and col["collateral"] != "not_stated":          # EVD-04
                fact["collateral"] = {"type": col["collateral"], "from_particulars": col["quote"]}
            facts[ref] = fact
    if fl := state.get("filings"):
        for f in fl["items"]:
            if f["category"] in ("accounts", "liquidation", "insolvency"):
                facts[filing_ref(f)] = {k: f[k] for k in ("date", "category", "description", "days_late")}
    if s.get("filings"):
        facts["computed"] = {"as_of": s["as_of"],
                             "months_since_accounts_made_up": s["filings"]["months_since_made_up"],
                             "months_since_incorporation": s["filings"]["months_since_incorporation"]}
    for a in state["applicable"]:
        facts[a["clause_id"]] = {"outcome": a["outcome"], "evidence": a["evidence"],
                                 "clause": CLAUSE[a["clause_id"]]["text"]}
    return facts

## 6 · The model's part — statements with sources

The answer is a list of statements, each with `sources`. It's the same pattern as 02's quote and 03's quotes,
except the evidence here is a *fact ID* rather than a quotation, because a brief summarises and doesn't copy.

The prompt tells the model the decision and clause table are written elsewhere, so it spends its sentences on
what the table can't say: what the facts *mean* for an approach. For example, a new facility would rank second
unless the existing charges are redeemed (SEC-01's text).

In [6]:
class Statement(BaseModel):
    text: str = Field(description="One or two plain-English sentences for the RM.")
    sources: list[str] = Field(description="Keys of the fact sheet this statement relies on, copied exactly.")


class Draft(BaseModel):
    statements: list[Statement]


BRIEF_SYSTEM = (
    "You write the summary of a brief for a bank relationship manager (RM) about one UK company, using a fact "
    "sheet. The decision and a table of the applicable policy clauses are added to the brief separately. Your job "
    "is 4 to 7 short statements telling the RM what matters: who the company is, its existing secured borrowing, "
    "its filing conduct, and what the applicable clauses mean for an approach.\n"
    "Rules:\n"
    "- Use only the fact sheet. No knowledge about lenders, sectors or markets, however accurate.\n"
    "- Every statement lists in sources the fact-sheet keys it relies on, copied exactly. A statement whose "
    "sources are not keys of the fact sheet will be deleted.\n"
    "- Say what a charge is secured on only if that charge's fact has a collateral entry.\n"
    "- When a statement relies on a policy clause, include the clause ID in its sources."
)

## 7 · Checks, rendering, and the brief node

`validate` is EVD-02 in code: a statement survives only if every source is a key of the fact sheet. `render`
assembles the brief, with code-written parts around the model's statements.

In [7]:
THIN = "Thin evidence — RM verification required"          # EVD-05, verbatim


def validate(draft: Draft | None, facts: dict) -> tuple[list[dict], list[dict]]:
    kept, dropped = [], []
    for st in (draft.statements if draft else []):
        unknown = [x for x in st.sources if x not in facts]
        item = {"text": st.text, "sources": st.sources}
        if not st.sources or unknown:
            dropped.append({**item, "why": f"unknown source(s) {unknown}" if unknown else "no source"})
        else:
            kept.append(item)
    return kept, dropped


def _cell(x) -> str:
    return str(x).replace("|", "\\|")


def render(state: CopilotState, kept: list[dict]) -> str:
    p, o, gaps = state.get("profile") or {}, state["outcome"], state["evidence_gap"]
    open_gaps = bool(gaps) and state["qualifies"] is None
    lines = [f"# RM brief — {p.get('company_name', '?')} ({state['company_number']})",
             f"*as of {state['signals']['as_of']} · research passes: {state.get('research_attempts', 0) + 1}*", ""]
    if open_gaps:                                                                    # EVD-05
        lines += [f"## ⚠ {THIN}", "", *[f"- {_cell(g)}" for g in gaps], ""]
    verdict = {True: "yes", False: "no", None: "not until the gaps are closed"}[state["qualifies"]]
    titles = [CLAUSE[c]["title"] if c in CLAUSE else c for c in o["clauses"]]
    lines += [f"**Decision: {o['decision']}** — set by {'; '.join(titles) or 'no clause'}  ",   # EVD-06
              f"**Qualifies as a lead:** {verdict}", ""]
    lines += ["## Summary", ""] + [f"- {st['text']} `[{'; '.join(st['sources'])}]`" for st in kept] + [""]
    lines += ["## Policy clauses that apply", "", "| clause | outcome | evidence |", "|---|---|---|"]    # EVD-07
    lines += [f"| {_cell(a['title'])} | {_cell(a['outcome'])} | {_cell('; '.join(a['evidence']))} |"
              for a in state["applicable"]]
    if gaps and not open_gaps:
        lines += ["", "## Evidence gaps (they cannot change this decision)", "", *[f"- {_cell(g)}" for g in gaps]]
    return "\n".join(lines)


async def brief(state: CopilotState) -> dict:
    facts = fact_sheet(state)
    resp = await llm.client().beta.messages.parse(
        model=llm.MODEL,
        max_tokens=16000,
        system=BRIEF_SYSTEM,
        messages=[{"role": "user", "content": json.dumps({"decision": state["outcome"], "facts": facts}, indent=1)}],
        output_format=Draft,
        **llm.FALLBACK,
    )
    print(llm.usage_line("brief", resp, len(facts), "facts"))
    ok = resp.stop_reason != "refusal" and resp.parsed_output is not None
    kept, dropped = validate(resp.parsed_output if ok else None, facts)
    cites = sorted({x for st in kept for x in st["sources"]} | {a["clause_id"] for a in state["applicable"]})
    return {"brief": render(state, kept), "citations": cites, "brief_dropped": dropped}

## 8 · The full graph

`build(tools)` takes the tool dict as an argument, so sections 11–12 can pass in a tool that fails on purpose.

In [8]:
def build(tools: dict):
    g = StateGraph(CopilotState)
    g.add_node("research", make_research(tools))
    g.add_node("signal", signal)
    g.add_node("policy", policy)
    g.add_node("supervisor", supervisor)
    g.add_node("brief", brief)
    g.add_edge(START, "research")
    g.add_edge("research", "signal")
    g.add_edge("signal", "policy")
    g.add_edge("policy", "supervisor")
    g.add_conditional_edges("supervisor", lambda s: s["route"], {"research": "research", "brief": "brief"})
    g.add_edge("brief", END)
    return g.compile()


graph = build(TOOL)
print(graph.get_graph().draw_ascii())


async def run(graph, number: str, show_brief: bool = True) -> dict:
    st = await graph.ainvoke({"company_number": number, "as_of": AS_OF})
    print("\nsupervisor_log:", *st["supervisor_log"], sep="\n  ")
    for d in st["brief_dropped"]:
        print(f"  dropped statement ({d['why']}): {d['text']}")
    if show_brief:
        display(Markdown(st["brief"]))
    return st

        +-----------+     
        | __start__ |     
        +-----------+     
               *          
               *          
               *          
         +----------+     
         | research |     
         +----------+     
          *        ..     
        **           .    
       *              ..  
+--------+              . 
| signal |              . 
+--------+              . 
     *                  . 
     *                  . 
     *                  . 
+--------+              . 
| policy |            ..  
+--------+           .    
          *        ..     
           **    ..       
             *  .         
        +------------+    
        | supervisor |    
        +------------+    
               .          
               .          
               .          
          +-------+       
          | brief |       
          +-------+       
               *          
               *          
               *          
         +---------+      
 

## 9 · Run it end to end — 36EL LTD (`10812571`)

From a company number to a brief in one call. Records come from the MCP server's disk cache, so the model calls are
the only cost: collateral (signal), none in policy, and one for the brief.

In [9]:
s36 = await run(graph, "10812571")

signal/collateral: 1 call · 1 text(s) · claude-opus-5 · 723 in / 61 out tokens


brief: 1 call · 21 facts · claude-opus-5 · 3870 in / 824 out tokens

supervisor_log:
  pass 1: DECLINE               → brief    no evidence gap


# RM brief — 36EL LTD (10812571)
*as of 2026-09-23 · research passes: 1*

**Decision: DECLINE** — set by CON-02 — Materially late filing  
**Qualifies as a lead:** no

## Summary

- 36EL LTD (company number 10812571) is an active private limited company incorporated on 09 June 2017, operating under SIC codes 68100 and 68209, with a single officer who has been the only appointment in the company's history. `[profile; officers]`
- The company holds two outstanding charges, both created on 10 July 2017 in favour of Interbay Funding Limited, each secured on the freehold interest in the land and property known as 36 Englands Lane, London NW3 4UE. `[108125710002; 108125710001]`
- Both charges contain fixed and floating elements and a negative pledge, so any new facility would rank behind Interbay unless redeemed, would need Interbay's written consent, and full redemption figures for both charges would be required before quoting a refinance. `[108125710002; 108125710001; SEC-01; SEC-06; SEC-07; SEC-08]`
- Filing conduct is poor and persistent: every one of the last seven accounts filings was late, ranging from 78 to 268 days, and the most recent micro-entity accounts (made up to 30 June 2024) were filed on 17 December 2025, 264 days late. `[AA filed 2025-12-17; AA filed 2024-06-27; AA filed 2023-06-30; AA filed 2022-11-08; AA filed 2021-07-04; AA filed 2020-12-21; AA filed 2019-06-14]`
- CON-02 drives the outcome: accounts filed more than 180 days after the statutory deadline are a material conduct failure, and the 264-day delay on the latest filing means the case is declined unless a documented explanation is held on file. `[CON-02; AA filed 2025-12-17]`
- Supporting conduct concerns reinforce the position — a pattern of repeated late filing, micro-entity accounts that disclose no turnover or profit, and financial information now 26.8 months old as at 23 September 2026, with accounts again showing as overdue. `[CON-01; CON-03; CON-04; CON-06; computed; profile]`
- If the customer approaches, do not indicate appetite; explain the filing record is the blocker and that any reconsideration would require a documented explanation for the 264-day delay plus management accounts covering the most recent 12 months. `[CON-02; CON-04; AA filed 2025-12-17]`

## Policy clauses that apply

| clause | outcome | evidence |
|---|---|---|
| CON-01 — Late filing of annual accounts | REFER | AA filed 2025-12-17 — 264 days late |
| CON-02 — Materially late filing | DECLINE | AA filed 2025-12-17 — 264 days late |
| CON-03 — Pattern of late filing | REFER | AA filed 2025-12-17; AA filed 2024-06-27 |
| CON-04 — Micro-entity accounts | REFER | AA filed 2025-12-17 — accounts with accounts type micro entity (made up date: 2024-06-30) |
| CON-06 — Stale financial information | REFER | accounts made up to 2024-06-30 — 26.8 months before as_of |
| SEC-01 — Outstanding third-party charge | REFER | 108125710002; 108125710001 |
| SEC-06 — Multiple simultaneous charges to one lender | REFER | 108125710001, 108125710002 (interbay funding limited) |
| SEC-07 — Negative pledge | REFER | 108125710002; 108125710001 |
| SEC-08 — Floating charge over the whole undertaking | REFER | 108125710002; 108125710001 |

## 10 · TESCO PLC (`00445790`) — gaps a retry can't close

Tesco leaves policy with four gaps: two from the truncated filing window, two from pre-2013 charges whose detail
is only an image. None came from a failed fetch, so the supervisor sends it straight to the brief (row 5) rather
than spending EVD-05's retries. The brief opens with the thin-evidence heading.

In [10]:
stesco = await run(graph, "00445790")

signal/collateral: 1 call · 2 text(s) · claude-opus-5 · 806 in / 132 out tokens


policy: 1 call · 4 question(s) · claude-opus-5 · 2449 in / 397 out tokens


brief: 1 call · 15 facts · claude-opus-5 · 3473 in / 1042 out tokens

supervisor_log:
  pass 1: REFER                 → brief    4 gap(s), none from a failed fetch — a retry would return the same record


# RM brief — TESCO PLC (00445790)
*as of 2026-09-23 · research passes: 1*

## ⚠ Thin evidence — RM verification required

- CON-01 (REFER) not determinable — filing window truncated; late accounts earlier in the 3 years may be missing
- CON-03 (REFER) not determinable — filing window truncated; only part of the 3 years was retrieved
- SEC-07 (REFER) not determinable — created 2009-11-04 · Tesco Trustee Company of Ir…: The negative pledge flag is null and the particulars text refers to an image I do not have, so whether a negative pledge exists cannot be established. \| created 2009-11-04 · Tesco Ireland Pension Trust…: The negative pledge flag is null and the particulars end with a reference to an unavailable image, leaving the clause condition unverifiable.
- SEC-08 (REFER) not determinable — created 2009-11-04 · Tesco Trustee Company of Ir…: Although the status is outstanding, the floating charge flag is null and the particulars point to an image not provided, so the presence of a floating charge over the whole undertaking cannot be determined. \| created 2009-11-04 · Tesco Ireland Pension Trust…: The floating charge indicator is null and the particulars refer to an image not supplied, so the clause's floating charge condition cannot be assessed despite the outstanding status.

**Decision: REFER** — set by SEC-01 — Outstanding third-party charge  
**Qualifies as a lead:** not until the gaps are closed

## Summary

- Tesco PLC (company number 00445790) is an active plc incorporated in November 1947, operating under SIC code 47110 (retail sale in non-specialised stores with food predominating). It has 11 active officers, with 74 having served in total and no new appointments in the last 12 months. `[profile; officers; computed]`
- Filing conduct is strong: the latest group accounts, made up to 28 February 2026, were filed on 25 July 2026, some 126 days ahead of the deadline, and nothing is overdue. The next accounts are not due until 26 August 2027. `[profile; AA filed 2026-07-25]`
- Two charges created on 4 November 2009 remain outstanding, both in favour of third-party pension trustees: Tesco Trustee Company of Ireland Limited and Tesco Ireland Pension Trustees Limited. `[created 2009-11-04 · Tesco Trustee Company of Ir…; created 2009-11-04 · Tesco Ireland Pension Trust…]`
- Each of those two outstanding charges is secured on a euro designated account in the company's name (sort code 40-05-15, account numbers 67851117 and 67851125) and the deposit held in it. `[created 2009-11-04 · Tesco Trustee Company of Ir…; created 2009-11-04 · Tesco Ireland Pension Trust…]`
- Because those outstanding charges are held by lenders outside the group, SEC-01 means we cannot offer a first-charge facility; any new facility would rank as a second charge unless the existing charges are redeemed at completion. This drives the REFER outcome. `[SEC-01; created 2009-11-04 · Tesco Trustee Company of Ir…; created 2009-11-04 · Tesco Ireland Pension Trust…]`
- A further seven charges, dating from 1991 to 2009 and including entries for Deutsche Bank AG, RBS Aerospace Limited and Cobroad Investments, are recorded as fully satisfied. Under SEC-03 these are disregarded for ranking and are only useful as evidence of prior borrowing capacity. `[SEC-03; created 2009-03-27 · Tesco Ireland Pension Trust…; created 2009-03-27 · Tesco Trustee Company of Ir…; created 2005-05-13 · Rbs Aerospace Limited; created 2001-07-31 · Deutsche International Fina…; created 2000-12-08 · Deutsche Bank Ag; created 1994-04-05 · Cobroad Investments; created 1991-12-17 · Cobroad Investments]`
- On approach, establish with the company whether the two 2009 pension-related charges can be released or redeemed at completion; if not, price and document on a second-charge basis. `[SEC-01; created 2009-11-04 · Tesco Trustee Company of Ir…; created 2009-11-04 · Tesco Ireland Pension Trust…]`

## Policy clauses that apply

| clause | outcome | evidence |
|---|---|---|
| SEC-01 — Outstanding third-party charge | REFER | created 2009-11-04 · Tesco Trustee Company of Ir…; created 2009-11-04 · Tesco Ireland Pension Trust… |
| SEC-03 — Satisfied charges disregarded | PROCEED | created 2009-03-27 · Tesco Ireland Pension Trust…; created 2009-03-27 · Tesco Trustee Company of Ir…; created 2005-05-13 · Rbs Aerospace Limited; created 2001-07-31 · Deutsche International Fina…; created 2000-12-08 · Deutsche Bank Ag; created 1994-04-05 · Cobroad Investments; created 1991-12-17 · Cobroad Investments |

## 11 · A fetch that fails once — the retry earns its keep

To test the loop without waiting for Companies House to have a bad day, wrap `get_charges` so its **first**
call raises, the way a timeout would. Everything else is the real server.

On pass 1, charges come back `None` with an error recorded. Policy then turns every SEC clause into a gap and EVD-01
applies, so the supervisor sees a failed fetch and retries. Pass 2 gets the real record, and Tesco ends where section 10
did.

In [11]:
class Flaky:
    # An MCP tool whose first `fails` calls raise, standing in for a network error.
    def __init__(self, tool, fails: int):
        self.tool, self.fails = tool, fails

    async def ainvoke(self, args):
        if self.fails > 0:
            self.fails -= 1
            raise ConnectionError("simulated: Companies House did not respond")
        return await self.tool.ainvoke(args)


s_once = await run(build({**TOOL, "get_charges": Flaky(TOOL["get_charges"], fails=1)}), "00445790",
                   show_brief=False)
print("\nfinal decision:", s_once["outcome"], "· same as section 10:", s_once["outcome"] == stesco["outcome"])

signal/collateral: 1 call · 2 text(s) · claude-opus-5 · 806 in / 132 out tokens


policy: 1 call · 4 question(s) · claude-opus-5 · 2449 in / 381 out tokens


brief: 1 call · 15 facts · claude-opus-5 · 3473 in / 862 out tokens

supervisor_log:
  pass 1: INSUFFICIENT EVIDENCE → research fetch failed (get_charges: ConnectionError: simulated: Companies House did not respond) — retry 1 of 2
  pass 2: REFER                 → brief    4 gap(s), none from a failed fetch — a retry would return the same record

final decision: {'decision': 'REFER', 'clauses': ['SEC-01']} · same as section 10: True


## 12 · A fetch that keeps failing — EVD-05's limit

Same wrapper, but it never recovers. The supervisor retries twice, then stops (row 4) and the brief goes out with
the thin-evidence heading and EVD-01's gap listed. Without a charge record there's no security position, so the
decision can't be better than INSUFFICIENT EVIDENCE.

In [12]:
s_down = await run(build({**TOOL, "get_charges": Flaky(TOOL["get_charges"], fails=99)}), "00445790")

brief: 1 call · 5 facts · claude-opus-5 · 1355 in / 540 out tokens

supervisor_log:
  pass 1: INSUFFICIENT EVIDENCE → research fetch failed (get_charges: ConnectionError: simulated: Companies House did not respond) — retry 1 of 2
  pass 2: INSUFFICIENT EVIDENCE → research fetch failed (get_charges: ConnectionError: simulated: Companies House did not respond) — retry 2 of 2
  pass 3: INSUFFICIENT EVIDENCE → brief    fetch still failing after 2 retries (EVD-05)


# RM brief — TESCO PLC (00445790)
*as of 2026-09-23 · research passes: 3*

## ⚠ Thin evidence — RM verification required

- CON-01 (REFER) not determinable — filing window truncated; late accounts earlier in the 3 years may be missing
- CON-03 (REFER) not determinable — filing window truncated; only part of the 3 years was retrieved
- SEC-01 (REFER) not determinable — charges record missing
- SEC-02 (PROCEED) not determinable — charges record missing
- SEC-03 (PROCEED) not determinable — charges record missing
- SEC-04 (REFER) not determinable — charges record missing
- SEC-05 (DECLINE) not determinable — charges record missing
- SEC-06 (REFER) not determinable — charges record missing
- SEC-07 (REFER) not determinable — charges record missing
- SEC-08 (REFER) not determinable — charges record missing
- EVD-01 missing: charges

**Decision: INSUFFICIENT EVIDENCE** — set by EVD-01 — Minimum evidence set  
**Qualifies as a lead:** not until the gaps are closed

## Summary

- TESCO PLC (company number 00445790) is an active plc incorporated on 27 November 1947, giving roughly 946 months of trading history, and is classified under SIC 47110 (retail sale in non-specialised stores with food, beverages or tobacco predominating). `[profile; computed]`
- The board is currently made up of 11 active officers out of 74 ever appointed, with no new appointments recorded in the last 12 months. `[officers]`
- Filing conduct is strong: the latest group accounts, made up to 28 February 2026, were filed on 25 July 2026 — some 126 days ahead of deadline — and no accounts are overdue, with the next set due 26 August 2027. `[AA filed 2026-07-25; profile]`
- As at the 23 September 2026 review date, the most recent accounts are only about 6.8 months old, so the financial picture is reasonably current. `[computed]`
- No charge data was retrieved for this company, so we cannot state what secured borrowing exists, on what collateral, or who the lenders are. `[EVD-01]`
- Clause EVD-01 requires a complete charge position — including the status and lender group of every charge — before any recommendation can be made; because that evidence is missing, the case returns for research rather than proceeding to a view. `[EVD-01]`
- Practically, do not approach the company on a lending proposition until the charge register has been pulled and the existing secured position is confirmed. `[EVD-01]`

## Policy clauses that apply

| clause | outcome | evidence |
|---|---|---|
| EVD-01 — Minimum evidence set | INSUFFICIENT EVIDENCE — return for research | charges |

## 13 · What the checks catch, and what they don't

| check | catches | doesn't catch |
|---|---|---|
| sources ⊆ fact sheet (EVD-02) | a statement citing nothing, or a record that doesn't exist | a statement that cites a real fact but **says more than the fact does** |
| collateral only if verified (EVD-04) | a collateral claim from an unverified label | a collateral claim with the wrong citation |
| code-written decision + table (EVD-06/07) | a misstated outcome or a missing clause, which can't happen by construction | — |

**The right-hand column happened in the run saved above.** Checking every summary statement by hand against the
records (the figures all hold, e.g. 36EL's "every one of the last seven accounts filings was late, ranging from 78
to 268 days" matches the filings exactly), two sentences go beyond their sources:

- **Tesco, section 10:** *"SIC code 47110 (retail sale in non-specialised stores with food predominating)"*. The fact
  sheet holds only the code `47110`; the description came from the model's general knowledge. That is what EVD-03
  forbids ("general knowledge about … a sector … must not appear in a brief, however accurate"), and it passed every
  check, because the statement cites `profile` and `profile` exists.
- **Tesco, section 12:** *"roughly 946 months of trading history"*. The fact is 945.8 months since *incorporation*;
  being incorporated is not the same as trading.

(Model wording varies between runs, so a re-run may produce different slips, or none.)

Both are **faithfulness** failures: the cited source is real, but the sentence says more than it does. Code can't
check that. It needs a reader: an evaluation with a second model as judge, scoring each statement against its cited
facts, run over a fixed set of companies. That's the natural next notebook. It's the same split as evaluating a RAG
system: the sources exist (checked here in code) is one question, the sentence is faithful to them is another.

The SIC slip also has a cheap fix at the source: put the SIC description on the fact sheet (Companies House
publishes the code list), so explaining the code *is* sourced.

In [13]:
print("citations in the 36EL brief:", s36["citations"])
print("statements dropped across all runs:",
      sum(len(s["brief_dropped"]) for s in (s36, stesco, s_once, s_down)))

citations in the 36EL brief: ['108125710001', '108125710002', 'AA filed 2019-06-14', 'AA filed 2020-12-21', 'AA filed 2021-07-04', 'AA filed 2022-11-08', 'AA filed 2023-06-30', 'AA filed 2024-06-27', 'AA filed 2025-12-17', 'CON-01', 'CON-02', 'CON-03', 'CON-04', 'CON-06', 'SEC-01', 'SEC-06', 'SEC-07', 'SEC-08', 'computed', 'officers', 'profile']
statements dropped across all runs: 0


## What comes next

The graph is complete: a company number goes in, and a cited brief comes out, with a retry loop that only fires when it can help.

| next | what | why |
|---|---|---|
| 05 · evaluation | an LLM-as-judge faithfulness score per brief statement, plus fixed expected outcomes for a set of companies | section 13: the one property code can't check |
| batch over the lead list | run `graph` over the top-K from the lead-scoring model | the product: a brief per prospect, DECLINEs filtered out |
| close the non-retryable gaps at the source | a date-bounded filing fetch in `mcp_ch/tools.py` (not `max_items=60`); `paper_filed` for CON-08 | turns Tesco-style gaps into answers instead of headings |